# Дообучение LUNA на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-22 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить LUNA (NeurIPS 2025, arXiv 2510.22257, код github.com/pulp-bio/BioFoundation) по той же схеме, что REVE, CBraMod, BIOT и DeepEpilepsy, и сравнить / собрать в ансамбль с REVE |
| **Веса** | HF `PulpBio/LUNA`, `LUNA_large.safetensors` (162,7 МБ, 40,4 млн параметров без головы реконструкции). Лицензия CC BY-ND 4.0: дообучать для себя можно, делиться изменёнными весами нельзя — веса никуда не сохраняются. **Предобучена на TUEG + Siena** (>21 000 ч): для будущей проверки на TUEP нужно исключать пересечение пациентов |
| **Выбор варианта** | large, а не base: у авторов на TUAB large чуть лучше (AUROC 0,892 против 0,887), а по вычислениям LUNA лёгкая (каналы сводятся к 6 запросам, затем 10 слоёв по 32 токенам на окно) — large укладывается в бюджет 1,5 ч GPU с выбором скорости обучения и 3 запусками. huge (311 млн) — не по бюджету |
| **Данные** | Обучение: NMT train (`mhmdrdwn/nmt-eeg`), до 16 окон по 5 с на запись (те же 80 с сигнала, что 8 окон по 10 с у REVE/BIOT), 15 % записей — выбор скорости обучения, эпохи и порога. Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`) |
| **Вход** | Как у авторов (`make_datasets/process_raw_eeg.py`, `tasks/finetune_task_LUNA.py`): 256 Гц, окна 5 с (32 патча по 40 отсчётов), фильтр 0,1–75 Гц + режекция (у нас 50 Гц), биполярный монтаж TCP как у TUEG (20 пар без A1/A2), координаты пары — середина двух электродов `standard_1005`, z-оценка каждого канала внутри окна (абсолютная амплитуда и соотношение амплитуд каналов пропадают — неверный масштаб NMT не важен) |
| **Аугментации** | зеркальная замена полушарий (p = 0,5), шум 0,1·СКО, зануление 1–2 каналов (p = 0,5), сдвиг до 2 с; дрожание координат ±0,02 при обучении — как у авторов («пустая» маска) |
| **Обучение** | вся модель + голова авторов (запрос-агрегатор + MLP) на 1 выход, AdamW (wd 0,05), послойное снижение скорости 0,75 как у авторов, разогрев 5 % + косинус, пакет 64, fp16 с проверкой NaN; базовая скорость выбирается по отложенным 15 % из 5·10⁻⁵ / 1,5·10⁻⁴ / 5·10⁻⁴ (5·10⁻⁴ — значение авторов), затем 3 запуска |
| **Среда** | Kaggle, GPU T4 |

In [ ]:
import os, sys, glob, time, types, subprocess, warnings
from fractions import Fraction
ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = False   # True — короткая проверка на процессоре (часть записей, 1 эпоха), тестовые цифры не смотреть
LUNA_COMMIT = "d4484feccf62eeec666380cb55b5d5b1d517cd71"   # BioFoundation, 12.10.2026 — код, по которому сверялась подготовка сигнала
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "rotary-embedding-torch==0.8.9", "einops==0.8.1"], check=True)
    if not os.path.exists("/kaggle/temp/BioFoundation"):
        subprocess.run(["git", "clone", "-q", "https://github.com/pulp-bio/BioFoundation.git", "/kaggle/temp/BioFoundation"], check=True)
        subprocess.run(["git", "-C", "/kaggle/temp/BioFoundation", "checkout", "-q", LUNA_COMMIT], check=True)
    LUNA_DIR, OUT = "/kaggle/temp/BioFoundation", "/kaggle/working"   # код авторов не в /kaggle/working, чтобы не попал в вывод
else:
    LUNA_DIR, OUT = os.environ["LUNA_DIR"], "."
sys.path.insert(0, LUNA_DIR)
# channel_embeddings.py импортирует torcheeg только ради списка каналов SEED (нужен при предобучении) — подменяем пустым модулем
for n in ["torcheeg", "torcheeg.datasets", "torcheeg.datasets.constants"]:
    sys.modules.setdefault(n, types.ModuleType(n))
sys.modules["torcheeg.datasets.constants"].SEED_CHANNEL_LIST = []
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*enable_nested_tensor.*")
import numpy as np, pandas as pd, torch, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from safetensors.torch import load_file
from models.LUNA import LUNA
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_NB = time.time()

SFREQ, SEG = 256, 5
L = SFREQ * SEG
CAP_TRAIN, CAP_EVAL = 16, 120   # окон по 5 с на запись: те же 80 с / 600 с сигнала, что 8 / 60 окон по 10 с у REVE и BIOT

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
# биполярный монтаж TCP, как TUEG_CHANNEL_LIST авторов, без A1-T3 и T4-A2 (ушных электродов в наших данных нет). Fz, Pz не используются
TCP = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"), ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),
       ("T3", "C3"), ("C3", "Cz"), ("Cz", "C4"), ("C4", "T4"),
       ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"), ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")]
PA = [EL.index(a) for a, _ in TCP]; PB = [EL.index(b) for _, b in TCP]
NCH = len(TCP)
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]   # на 19 исходных каналах, до вычитания пар

if ON_KAGGLE:
    # как get_channel_locations авторов: standard_1005 через info.set_montage, у пары — середина двух электродов
    import mne
    mne.set_log_level("ERROR")
    names = sorted({e for p in TCP for e in p})
    info = mne.create_info(names, SFREQ, "eeg")
    info.set_montage(mne.channels.make_standard_montage("standard_1005"), match_case=False)
    cp = info.get_montage().get_positions()["ch_pos"]
    POS = np.stack([(cp[a] + cp[b]) / 2 for a, b in TCP])
else:
    POS = np.random.default_rng(0).random((NCH, 3))   # только для проверки кода без mne
POS = torch.tensor(POS, dtype=torch.float32, device=DEV).unsqueeze(0)
print("координаты пар:", tuple(POS.shape[1:]), "| разброс по осям (м):", np.round((POS[0].max(0).values - POS[0].min(0).values).cpu().numpy(), 3))

SIZE = os.environ.get("LUNA_SIZE", "large")
CFG = {"base": dict(patch_size=40, embed_dim=64, num_heads=2, depth=8, num_queries=4, drop_path=0.1),     # config/model/LUNA_base.yaml
       "large": dict(patch_size=40, embed_dim=96, num_heads=2, depth=10, num_queries=6, drop_path=0.1)}[SIZE]   # config/model/LUNA_large.yaml
if ON_KAGGLE:
    from huggingface_hub import hf_hub_download
    WEIGHTS = hf_hub_download("PulpBio/LUNA", f"LUNA_{SIZE}.safetensors", cache_dir="/kaggle/temp/hf")   # не в /kaggle/working: веса не попадают в вывод
else:
    WEIGHTS = os.environ["LUNA_W"]
SD = {k: v for k, v in load_file(WEIGHTS).items() if not k.startswith(("decoder_head", "channel_emb"))}   # голова реконструкции не нужна

def new_luna(seed):
    torch.manual_seed(seed)
    m = LUNA(num_classes=1, **CFG)
    r = m.load_state_dict(SD, strict=False)
    assert not r.unexpected_keys and r.missing_keys and all(k.startswith("classifier.") for k in r.missing_keys), r
    return m.to(DEV)

NOISE = 0.1
def batch_input(W, g=None):
    """19 каналов -> (зеркало) -> 20 пар TCP -> z-оценка каждого канала в окне (как ChannelWiseNormalize авторов) -> (шум, зануление, сдвиг)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    if g is not None:
        m = torch.from_numpy(g.random(len(x)) < 0.5).to(DEV)
        x[m] = x[m][:, MIRROR]
    x = x[:, PA] - x[:, PB]
    x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-8)
    if g is not None:
        x = x + NOISE * torch.randn_like(x)
        for b in range(len(x)):
            if g.random() < 0.5:
                x[b, g.choice(NCH, g.integers(1, 3), replace=False)] = 0
            x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")
def fwd(model, x, train=False):
    # у авторов в дообучении передаётся «пустая» маска — она включает дрожание координат ±0,02; при оценке — без него (детерминированно)
    mask = torch.zeros_like(x, dtype=torch.bool) if train else None
    return model(x, mask, POS.expand(len(x), -1, -1))[0].squeeze(-1)

def predict(model, Ws, bs=256):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(fwd(model, batch_input(W[b:b + bs])).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.1, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def lu_filter(X):
    """0,1-75 Гц + режекция 50 Гц (у авторов 0,1-75 Гц + 60 Гц для американских TUH)."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def to_256(X, sf):
    fr = Fraction(SFREQ, int(round(sf))).limit_denominator(1000)
    return X if fr == 1 else ss.resample_poly(X, fr.numerator, fr.denominator, axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор", f"| LUNA-{SIZE}:",
      sum(p.numel() for p in new_luna(0).parameters()), "параметров")

## 2. NMT и турецкий набор (подкачка обоих параллельно)

In [ ]:
import mne
mne.set_log_level("ERROR")
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
if SMOKE:
    lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                     lab[(lab["loc"] == "eval")].head(20)]).sort_values("recordname").reset_index(drop=True)
    tfiles = tfiles[:10] + tfiles[-10:]
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time(); allf = list(lab.path) + [f for f, _ in tfiles]
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, allf)):
        if (i + 1) % 500 == 0 or i + 1 == len(allf):
            print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

NMT_STORE_K = 100.0   # масштаб в файлах неверный; каналы всё равно z-нормируются, множитель только для float16
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
        X = to_256(X, raw.info["sfreq"])
    except Exception:
        return r.recordname, None
    W = windows(lu_filter(X))
    if len(W) == 0:
        return r.recordname, None
    if r["loc"] == "train" and len(W) > CAP_TRAIN:
        W = W[np.sort(np.random.default_rng(i).choice(len(W), CAP_TRAIN, replace=False))]
    elif r["loc"] == "eval":
        W = W[spread(len(W), CAP_EVAL)]
    return r.recordname, W.astype(np.float16)

NMT = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон по 5 с: train", sum(len(NMT[n]) for n in tr.recordname), "| eval", sum(len(NMT[n]) for n in ev.recordname))

## 3. Турецкий набор: первые 19 каналов (относительно уха) → средний референт, 500→256 Гц, куски по 15 с подряд, окна 5 с

In [ ]:
# индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)          # средний референт (для биполярных пар не важен), как в образце — для СКО записи
    X = to_256(X, 500)                        # 500 -> 256 Гц, кусок 15 с = 3840 отсчётов = 3 окна
    W = windows(lu_filter(np.concatenate(list(X), axis=-1)))
    return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))

TR, TR_AMP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
        TR.append(a); TR_AMP.append(b)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR]))}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение LUNA

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
Xw = np.concatenate([NMT[n] for n in fit_names])
val_W = [NMT[n] for n in val_names]
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)}")

BS, EPOCHS, LLRD, WD = 64, (1 if SMOKE else 6), 0.75, 0.05
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()

def make_opt(model, total_steps, lr):
    """AdamW, послойное снижение скорости как в configure_optimizers авторов: blocks.i -> lr * 0.75^(depth - i), остальное -> lr."""
    groups = {}
    for n, p in model.named_parameters():
        if p.requires_grad:
            k = LLRD ** (CFG["depth"] - int(n.split(".")[1])) if n.startswith("blocks.") else 1.0
            groups.setdefault(k, []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": lr * k} for k, ps in groups.items()], weight_decay=WD, betas=(0.9, 0.999))
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t):
    with torch.autocast(**AMP):
        out = fwd(model, x, train=True)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

def train_run(seed, lr, epochs=EPOCHS, tag="luna"):
    g = np.random.default_rng(seed)
    model = new_luna(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, epochs * steps_ep, lr)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad = -1, None, 0, [], 0
    t0 = time.time()
    for ep in range(epochs):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            l, bad = step(model, opt, sched, scaler, batch_input(Xw[j], g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(auc, 3)))
        print(f"  {tag} lr {lr:g} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_ep, best_auc, hist, n_bad

In [ ]:
# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны убывать и уйти ниже 0,69 ---
g0 = np.random.default_rng(123)
n_sm, ep_sm = (32, 3) if SMOKE else (128, 8)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], n_sm, replace=False) for c in (0, 1)])
model = new_luna(0); opt, sched = make_opt(model, ep_sm * (len(sm) // BS), 1.5e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
model.train(); sm_losses, tt = [], []
for ep in range(ep_sm):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, scaler, batch_input(Xw[j]), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:])) if len(tt) > 3 else float(np.median(tt))
del model, opt
if DEV == "cuda": torch.cuda.empty_cache()
CONVERGED = bool(np.isfinite(sm_losses).all() and np.mean(sm_losses[-2:]) < np.mean(sm_losses[:2]) - 0.05 and min(sm_losses) < 0.66)
print("сходимость на подвыборке:", "есть" if CONVERGED else f"НЕТ ({np.round(sm_losses, 3)}) — продолжаю, скорость выбирается по отложенным 15 %")
est_run = (EPOCHS * (len(yw) / BS) * t_step * 1.5 + EPOCHS * 20) / 60   # мин (+50 % на аугментации, ~20 с на выбор эпохи)
print(f"шаг {t_step:.3f} с | оценка одного запуска {est_run:.1f} мин")

In [ ]:
# --- выбор базовой скорости обучения: только по отложенным 15 % NMT train (запуск 0) ---
TIME_LIMIT_MIN = 50   # бюджет задачи 1,5 ч GPU; чтение данных ~25 мин
T_TRAIN, GRID = time.time(), {}
for lr in ([1.5e-4] if SMOKE else [5e-5, 1.5e-4, 5e-4]):
    if (time.time() - T_TRAIN) / 60 + est_run > 0.6 * TIME_LIMIT_MIN:
        print(f"ПРОПУСК lr {lr:g}: не укладывается"); continue
    model, best_ep, best_auc, hist, n_bad = train_run(0, lr, tag="выбор")
    GRID[lr] = dict(model=model.cpu(), best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad)
    print(f"lr {lr:g}: лучшая эпоха {best_ep}, отложенные NMT {best_auc:.3f}", flush=True)
LR = max(GRID, key=lambda k: GRID[k]["best_auc"])
print("выбрана скорость:", LR)
pd.DataFrame([dict(lr=k, best_epoch=v["best_ep"], val=round(v["best_auc"], 3), hist=v["hist"], nan_steps=v["n_bad"]) for k, v in GRID.items()]) \
  .to_csv(f"{OUT}/luna_lr_choice.csv", index=False)
for k in [k for k in GRID if k != LR]: del GRID[k]["model"]

# --- 3 запуска с выбранной скоростью (запуск 0 — уже обученный при выборе); тестовые наборы — один раз ---
RUNS, dur = [], []
for seed in ([0] if SMOKE else [0, 1, 2]):
    t0 = time.time()
    if seed == 0:
        G = GRID[LR]; model, best_ep, best_auc, hist, n_bad = G["model"].to(DEV), G["best_ep"], G["best_auc"], G["hist"], G["n_bad"]
    else:
        est = np.mean(dur) if dur else est_run * 60
        if (time.time() - T_TRAIN + est) / 60 > TIME_LIMIT_MIN:
            print(f"ПРОПУСК запуск {seed}: не укладывается в {TIME_LIMIT_MIN} мин"); continue
        model, best_ep, best_auc, hist, n_bad = train_run(seed, LR)
    s_ev = predict(model, [NMT[n] for n in ev.recordname]); s_tr = predict(model, TR)
    dur.append(time.time() - t0)
    RUNS.append(dict(model="luna", lr=LR, seed=seed, best_epoch=best_ep, val=round(best_auc, 3), hist=hist, nan_steps=n_bad,
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr)), s_val=predict(model, val_W)))
    print(f"luna запуск {seed}: эпоха {best_ep} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
          f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)}", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
del GRID
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RUNS])
runs.to_csv(f"{OUT}/luna_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги: ROC AUC, пороговые метрики (порог — по отложенным 15 % NMT train, на Турцию перенесён тот же)

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "LUNA-large дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "LUNA-large дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv(f"{OUT}/luna_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков (у рангов нет общего масштаба между наборами)
p_val = np.mean([r["s_val"] for r in RUNS], 0); p_ev = np.mean([r["s_ev"] for r in RUNS], 0); p_tr = np.mean([r["s_tr"] for r in RUNS], 0)
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "LUNA", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "NMT eval" else "; перенесён"), "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{OUT}/luna_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
for d, k in [(d1, "s_ev"), (d2, "s_tr")]:
    for r in RUNS: d[f"luna_seed{r['seed']}"] = r[k]
d1["luna_ens_rank"], d2["luna_ens_rank"] = ens_ev, ens_tr
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_names), "y": y_val})
for r in RUNS: d3[f"luna_seed{r['seed']}"] = r["s_val"]
pd.concat([d1, d2, d3]).to_csv(f"{OUT}/luna_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (без строк о подкачке файлов и служебных предупреждений)

Тестовые наборы считались один раз. NaN/inf нет, потери падают (0,67 → 0,58), у групп вероятности различны (119 разных значений из 121). Перед этим блокнот прошёл короткую проверку на процессоре Kaggle (`SMOKE = True`: 60 записей NMT train, 20 eval, 20 турецких, 1 эпоха; GPU не тратился); её тестовые цифры для решений не использовались.

```
координаты пар: (20, 3) | разброс по осям (м): [0.158 0.164 0.096]
устройство: Tesla T4 | LUNA-large: 43111274 параметров
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT прочитано 400/2417 161 с
NMT прочитано 800/2417 320 с
NMT прочитано 1200/2417 480 с
NMT прочитано 1600/2417 661 с
NMT прочитано 2000/2417 830 с
NMT прочитано 2400/2417 1022 с
NMT прочитано 2417/2417 1031 с
label  abnormal  normal
loc                    
eval         90      95
train       325    1905
окон по 5 с: train 35558 | eval 20829
Турция прочитано за 143 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 120
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 30199 (патология 0.14) | записей на выбор: 335
  проверка: эпоха 1 потери 0.769
  проверка: эпоха 2 потери 0.694
  проверка: эпоха 3 потери 0.761
  проверка: эпоха 4 потери 0.741
  проверка: эпоха 5 потери 0.665
  проверка: эпоха 6 потери 0.655
  проверка: эпоха 7 потери 0.687
  проверка: эпоха 8 потери 0.657
сходимость на подвыборке: есть
шаг 0.135 с | оценка одного запуска 11.5 мин
  выбор lr 5e-05 запуск 0 | эпоха 1: потери 0.659 | отложенные NMT 0.723 | шагов с NaN/inf 0 | 77 с
  выбор lr 5e-05 запуск 0 | эпоха 2: потери 0.632 | отложенные NMT 0.795 | шагов с NaN/inf 0 | 153 с
  выбор lr 5e-05 запуск 0 | эпоха 3: потери 0.613 | отложенные NMT 0.815 | шагов с NaN/inf 0 | 230 с
  выбор lr 5e-05 запуск 0 | эпоха 4: потери 0.599 | отложенные NMT 0.828 | шагов с NaN/inf 0 | 307 с
  выбор lr 5e-05 запуск 0 | эпоха 5: потери 0.590 | отложенные NMT 0.837 | шагов с NaN/inf 0 | 383 с
  выбор lr 5e-05 запуск 0 | эпоха 6: потери 0.582 | отложенные NMT 0.833 | шагов с NaN/inf 0 | 459 с
lr 5e-05: лучшая эпоха 5, отложенные NMT 0.837
  выбор lr 0.00015 запуск 0 | эпоха 1: потери 0.669 | отложенные NMT 0.738 | шагов с NaN/inf 0 | 76 с
  выбор lr 0.00015 запуск 0 | эпоха 2: потери 0.632 | отложенные NMT 0.805 | шагов с NaN/inf 0 | 153 с
  выбор lr 0.00015 запуск 0 | эпоха 3: потери 0.620 | отложенные NMT 0.812 | шагов с NaN/inf 0 | 230 с
  выбор lr 0.00015 запуск 0 | эпоха 4: потери 0.603 | отложенные NMT 0.837 | шагов с NaN/inf 0 | 306 с
  выбор lr 0.00015 запуск 0 | эпоха 5: потери 0.591 | отложенные NMT 0.837 | шагов с NaN/inf 0 | 383 с
  выбор lr 0.00015 запуск 0 | эпоха 6: потери 0.579 | отложенные NMT 0.841 | шагов с NaN/inf 0 | 460 с
lr 0.00015: лучшая эпоха 6, отложенные NMT 0.841
  выбор lr 0.0005 запуск 0 | эпоха 1: потери 0.688 | отложенные NMT 0.710 | шагов с NaN/inf 0 | 77 с
  выбор lr 0.0005 запуск 0 | эпоха 2: потери 0.645 | отложенные NMT 0.728 | шагов с NaN/inf 0 | 155 с
  выбор lr 0.0005 запуск 0 | эпоха 3: потери 0.634 | отложенные NMT 0.740 | шагов с NaN/inf 0 | 232 с
  выбор lr 0.0005 запуск 0 | эпоха 4: потери 0.625 | отложенные NMT 0.763 | шагов с NaN/inf 0 | 309 с
  выбор lr 0.0005 запуск 0 | эпоха 5: потери 0.614 | отложенные NMT 0.781 | шагов с NaN/inf 0 | 386 с
  выбор lr 0.0005 запуск 0 | эпоха 6: потери 0.603 | отложенные NMT 0.783 | шагов с NaN/inf 0 | 464 с
lr 0.0005: лучшая эпоха 6, отложенные NMT 0.783
выбрана скорость: 0.00015
luna запуск 0: эпоха 6 | NMT eval 0.856 | Турция 0.769 | медиана p здоровые/эпилепсия 0.109/0.239 | разных значений 119/121
  luna lr 0.00015 запуск 1 | эпоха 1: потери 0.669 | отложенные NMT 0.745 | шагов с NaN/inf 0 | 77 с
  luna lr 0.00015 запуск 1 | эпоха 2: потери 0.641 | отложенные NMT 0.759 | шагов с NaN/inf 0 | 154 с
  luna lr 0.00015 запуск 1 | эпоха 3: потери 0.620 | отложенные NMT 0.782 | шагов с NaN/inf 0 | 230 с
  luna lr 0.00015 запуск 1 | эпоха 4: потери 0.606 | отложенные NMT 0.809 | шагов с NaN/inf 0 | 307 с
  luna lr 0.00015 запуск 1 | эпоха 5: потери 0.595 | отложенные NMT 0.812 | шагов с NaN/inf 0 | 384 с
  luna lr 0.00015 запуск 1 | эпоха 6: потери 0.579 | отложенные NMT 0.821 | шагов с NaN/inf 0 | 461 с
luna запуск 1: эпоха 6 | NMT eval 0.856 | Турция 0.815 | медиана p здоровые/эпилепсия 0.088/0.226 | разных значений 119/121
  luna lr 0.00015 запуск 2 | эпоха 1: потери 0.666 | отложенные NMT 0.733 | шагов с NaN/inf 0 | 77 с
  luna lr 0.00015 запуск 2 | эпоха 2: потери 0.634 | отложенные NMT 0.770 | шагов с NaN/inf 0 | 154 с
  luna lr 0.00015 запуск 2 | эпоха 3: потери 0.612 | отложенные NMT 0.814 | шагов с NaN/inf 0 | 232 с
  luna lr 0.00015 запуск 2 | эпоха 4: потери 0.598 | отложенные NMT 0.820 | шагов с NaN/inf 0 | 309 с
  luna lr 0.00015 запуск 2 | эпоха 5: потери 0.589 | отложенные NMT 0.825 | шагов с NaN/inf 0 | 386 с
  luna lr 0.00015 запуск 2 | эпоха 6: потери 0.578 | отложенные NMT 0.835 | шагов с NaN/inf 0 | 463 с
luna запуск 2: эпоха 6 | NMT eval 0.850 | Турция 0.791 | медиана p здоровые/эпилепсия 0.084/0.195 | разных значений 119/121
всё обучение: 40.1 мин
model  lr  seed  best_epoch   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
 luna 0.0     0           6 0.841          0     0.856   0.769          0.109          0.239          119
 luna 0.0     1           6 0.821          0     0.856   0.815          0.088          0.226          119
 luna 0.0     2           6 0.835          0     0.850   0.791          0.084          0.195          119
                                   модель n          NMT eval                             Турция
          LUNA-large дообученная, запуски 3     0.854 ± 0.004 0.792 ± 0.023 (0.77 / 0.81 / 0.79)
LUNA-large дообученная, ансамбль запусков   0.855 (0.80-0.90)                  0.795 (0.71-0.87)
Связь с амплитудой записи на Турции (Спирмен): 0.19
модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
  LUNA                  Юден            0.526 NMT eval 185 0.75 (0.69-0.81)  0.75 (0.68-0.81) 0.78 (0.68-0.86) 0.69 (0.59-0.78) 0.81 (0.73-0.89)
  LUNA                  Юден            0.526   Турция 121 0.64 (0.55-0.72)  0.56 (0.52-0.61) 1.00 (1.00-1.00) 0.12 (0.04-0.21) 1.00 (1.00-1.00)
  LUNA чувствительность 80 %            0.506 NMT eval 185 0.74 (0.66-0.79)  0.73 (0.66-0.80) 0.75 (0.65-0.84) 0.69 (0.59-0.78) 0.78 (0.69-0.86)
  LUNA чувствительность 80 %            0.506   Турция 121 0.64 (0.55-0.72)  0.56 (0.52-0.61) 1.00 (1.00-1.00) 0.12 (0.04-0.21) 1.00 (1.00-1.00)
всё время блокнота: 60.9 мин
```

## 7. Сравнение с другими моделями, ансамбль с REVE, калибровка на месте (локально, без GPU, по сохранённым предсказаниям)

Запускать из папки блокнота: читает `results/luna_finetune_per_person.csv` и `results/biot_ensembles_per_person.csv` (там предсказания REVE, CBraMod, BIOT, DeepEpilepsy по людям).

In [ ]:
# Сравнение LUNA с другими моделями по сохранённым предсказаниям (без GPU): ансамбль с REVE, парные разности, калибровка на месте
import sys, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
lu = pd.read_csv(f"{RES}/luna_finetune_per_person.csv")
ot = pd.read_csv(f"{RES}/biot_ensembles_per_person.csv")
lu_cols = [c for c in lu.columns if c.startswith("luna_seed")]
m = lu[lu.dataset.isin(["nmt_eval", "turkey"])].merge(
    ot[["dataset", "id", "y", "reve_ens_rank", "cb_ens_rank", "de_ens_rank", "biot_ens_rank"]], on=["dataset", "id"], suffixes=("", "_o"))
assert (m.y == m.y_o).all()
print({d: int((m.dataset == d).sum()) for d in ["nmt_eval", "turkey"]}, "общих записей")

rk = lambda s: rankdata(s) / len(s)
SC = {}
for ds in ["nmt_eval", "turkey"]:
    d = m[m.dataset == ds]
    s = {"LUNA": np.mean([rk(d[c]) for c in lu_cols], 0), "REVE": rk(d.reve_ens_rank), "CBraMod": rk(d.cb_ens_rank),
         "BIOT": rk(d.biot_ens_rank), "DeepEpilepsy": rk(d.de_ens_rank)}
    s["LUNA + REVE"] = (s["LUNA"] + s["REVE"]) / 2
    s["LUNA + REVE + CBraMod"] = (s["LUNA"] + s["REVE"] + s["CBraMod"]) / 3
    s["LUNA + REVE + DeepEpilepsy"] = (s["LUNA"] + s["REVE"] + s["DeepEpilepsy"]) / 3
    SC[ds] = (d.y.to_numpy(), s, d)

def boot(y, f, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(f(i))
    return np.percentile(v, [2.5, 97.5])

rows = []
for name in SC["turkey"][1]:
    row = {"модель / ансамбль (средние ранги)": name}
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]; lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[name][i]))
        row[lab] = f"{roc_auc_score(y, s[name]):.3f} ({lo:.2f}-{hi:.2f})"
    rows.append(row)
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ):")
diffs = []
for a, b in [("LUNA", "REVE"), ("LUNA", "CBraMod"), ("LUNA", "BIOT"), ("LUNA", "DeepEpilepsy"), ("LUNA + REVE", "REVE"), ("LUNA + REVE", "LUNA")]:
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]
        f = lambda i: roc_auc_score(y[i], s[a][i]) - roc_auc_score(y[i], s[b][i])
        lo, hi = boot(y, f); dv = f(np.arange(len(y)))
        diffs.append({"разность": f"{a} − {b}", "набор": lab, "ΔROC AUC": round(dv, 3), "ДИ": f"{lo:+.3f}…{hi:+.3f}"})
        print(f"  {a} − {b} | {lab}: {dv:+.3f} ({lo:+.3f}…{hi:+.3f})")
y, s, d = SC["turkey"]
print("\nСпирмен на Турции: LUNA–амплитуда", round(spearmanr(s["LUNA"], d.amp).correlation, 2),
      "| LUNA–REVE", round(spearmanr(s["LUNA"], s["REVE"]).correlation, 2), "| LUNA–CBraMod", round(spearmanr(s["LUNA"], s["CBraMod"]).correlation, 2),
      "| LUNA–BIOT", round(spearmanr(s["LUNA"], s["BIOT"]).correlation, 2), "| LUNA–DeepEpilepsy", round(spearmanr(s["LUNA"], s["DeepEpilepsy"]).correlation, 2))
print("одна амплитуда как оценка, ROC AUC:", round(roc_auc_score(y, d.amp), 3))

# --- калибровка на месте: порог по k размеченным людям Турции (поровну классов, индекс Юдена), проверка на остальных; 500 повторов, одни и те же выборки для всех моделей ---
def calib(y, sc, k, reps=500, seed=0):
    r = np.random.default_rng(seed); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]; M = {n: [] for n in sc}
    for _ in range(reps):
        c = np.concatenate([r.choice(i1, k // 2, replace=False), r.choice(i0, k // 2, replace=False)])
        rest = np.setdiff1d(np.arange(len(y)), c); yy = y[rest]
        for n, s in sc.items():
            fpr, tpr, th = roc_curve(y[c], s[c]); pr = s[rest] >= th[np.argmax(tpr - fpr)]
            tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
            rec, spec = tp / (tp + fn), tn / (tn + fp)
            M[n].append(dict(rec=rec, spec=spec, bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(yy)))
    out = []
    for n in sc:
        D = pd.DataFrame(M[n]); f = lambda c: f"{D[c].mean():.2f} ({np.nanpercentile(D[c], 2.5):.2f}–{np.nanpercentile(D[c], 97.5):.2f})"
        out.append({"калибровка": f"{k} чел.", "модель": n, "recall": f("rec"), "специфичность": f("spec"), "сбаланс. accuracy": f("bacc"),
                    "precision": f("prec"), "accuracy": f("acc")})
    return out
cal_models = ["LUNA", "REVE", "CBraMod", "BIOT", "DeepEpilepsy", "LUNA + REVE"]
cal = pd.DataFrame([row for k in [10, 20, 30] for row in calib(y, {n: s[n] for n in cal_models}, k)])
print("\nКалибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):")
print(cal.to_string(index=False))

summ.to_csv(f"{RES}/luna_ensembles_summary.csv", index=False)
pd.DataFrame(diffs).to_csv(f"{RES}/luna_paired_differences.csv", index=False)
cal.to_csv(f"{RES}/luna_turkey_local_threshold_calibration.csv", index=False)
pp = []
for ds in ["nmt_eval", "turkey"]:
    yy, s, d = SC[ds]; t = d[["dataset", "id", "y"]].copy()
    for n, v in s.items(): t[n.replace(" + ", "_").replace(" ", "").lower() + "_rank"] = v
    pp.append(t)
pd.concat(pp).to_csv(f"{RES}/luna_ensembles_per_person.csv", index=False)


Вывод (выполнено локально):

```
{'nmt_eval': 184, 'turkey': 121} общих записей
модель / ансамбль (средние ранги)          NMT eval            Турция
                             LUNA 0.858 (0.80-0.91) 0.795 (0.71-0.87)
                             REVE 0.879 (0.83-0.92) 0.874 (0.80-0.93)
                          CBraMod 0.845 (0.79-0.90) 0.820 (0.74-0.88)
                             BIOT 0.798 (0.73-0.86) 0.821 (0.73-0.89)
                     DeepEpilepsy 0.900 (0.85-0.94) 0.682 (0.59-0.77)
                      LUNA + REVE 0.894 (0.85-0.93) 0.852 (0.77-0.91)
            LUNA + REVE + CBraMod 0.887 (0.84-0.93) 0.863 (0.79-0.92)
       LUNA + REVE + DeepEpilepsy 0.905 (0.86-0.94) 0.840 (0.76-0.90)

Парный бутстреп разности ROC AUC (95% ДИ):
  LUNA − REVE | NMT eval: -0.021 (-0.073…+0.030)
  LUNA − REVE | Турция: -0.079 (-0.154…-0.007)
  LUNA − CBraMod | NMT eval: +0.012 (-0.040…+0.066)
  LUNA − CBraMod | Турция: -0.024 (-0.106…+0.056)
  LUNA − BIOT | NMT eval: +0.060 (+0.014…+0.108)
  LUNA − BIOT | Турция: -0.025 (-0.101…+0.046)
  LUNA − DeepEpilepsy | NMT eval: -0.042 (-0.087…-0.001)
  LUNA − DeepEpilepsy | Турция: +0.114 (+0.016…+0.212)
  LUNA + REVE − REVE | NMT eval: +0.015 (-0.009…+0.042)
  LUNA + REVE − REVE | Турция: -0.022 (-0.063…+0.015)
  LUNA + REVE − LUNA | NMT eval: +0.037 (+0.009…+0.068)
  LUNA + REVE − LUNA | Турция: +0.057 (+0.015…+0.100)

Спирмен на Турции: LUNA–амплитуда 0.19 | LUNA–REVE 0.69 | LUNA–CBraMod 0.64 | LUNA–BIOT 0.67 | LUNA–DeepEpilepsy 0.5
одна амплитуда как оценка, ROC AUC: 0.764

Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):
калибровка       модель           recall    специфичность сбаланс. accuracy        precision         accuracy
   10 чел.         LUNA 0.55 (0.13–0.89) 0.82 (0.36–1.00)  0.69 (0.56–0.73) 0.77 (0.49–1.00) 0.71 (0.59–0.77)
   10 чел.         REVE 0.62 (0.24–0.91) 0.88 (0.50–1.00)  0.75 (0.62–0.81) 0.85 (0.55–1.00) 0.78 (0.66–0.85)
   10 чел.      CBraMod 0.61 (0.18–0.96) 0.76 (0.36–1.00)  0.69 (0.59–0.74) 0.70 (0.51–1.00) 0.70 (0.60–0.74)
   10 чел.         BIOT 0.59 (0.14–0.92) 0.83 (0.36–0.98)  0.71 (0.56–0.77) 0.77 (0.49–0.92) 0.73 (0.59–0.80)
   10 чел. DeepEpilepsy 0.54 (0.02–0.94) 0.65 (0.22–1.00)  0.59 (0.51–0.65) 0.55 (0.45–0.80) 0.60 (0.52–0.64)
   10 чел.  LUNA + REVE 0.59 (0.21–0.93) 0.87 (0.48–1.00)  0.73 (0.60–0.80) 0.84 (0.55–1.00) 0.75 (0.66–0.82)
   20 чел.         LUNA 0.56 (0.25–0.93) 0.82 (0.34–1.00)  0.69 (0.61–0.74) 0.76 (0.48–1.00) 0.72 (0.57–0.78)
   20 чел.         REVE 0.65 (0.38–0.90) 0.88 (0.49–1.00)  0.77 (0.66–0.82) 0.84 (0.54–1.00) 0.79 (0.65–0.86)
   20 чел.      CBraMod 0.66 (0.25–0.95) 0.73 (0.39–1.00)  0.69 (0.61–0.74) 0.66 (0.51–1.00) 0.70 (0.61–0.74)
   20 чел.         BIOT 0.64 (0.34–0.88) 0.82 (0.43–0.98)  0.73 (0.63–0.78) 0.75 (0.50–0.93) 0.75 (0.60–0.81)
   20 чел. DeepEpilepsy 0.64 (0.10–0.95) 0.57 (0.25–0.97)  0.60 (0.52–0.66) 0.51 (0.44–0.71) 0.60 (0.52–0.65)
   20 чел.  LUNA + REVE 0.62 (0.33–0.93) 0.87 (0.48–1.00)  0.74 (0.66–0.80) 0.83 (0.53–1.00) 0.77 (0.64–0.84)
   30 чел.         LUNA 0.56 (0.29–0.91) 0.83 (0.36–1.00)  0.69 (0.63–0.75) 0.76 (0.47–1.00) 0.73 (0.57–0.80)
   30 чел.         REVE 0.67 (0.43–0.91) 0.87 (0.46–1.00)  0.77 (0.68–0.83) 0.82 (0.51–1.00) 0.79 (0.63–0.87)
   30 чел.      CBraMod 0.68 (0.26–0.94) 0.72 (0.51–1.00)  0.70 (0.62–0.76) 0.64 (0.53–1.00) 0.70 (0.65–0.76)
   30 чел.         BIOT 0.64 (0.40–0.86) 0.82 (0.62–0.98)  0.73 (0.66–0.79) 0.74 (0.57–0.93) 0.75 (0.69–0.81)
   30 чел. DeepEpilepsy 0.66 (0.14–0.94) 0.55 (0.23–0.95)  0.61 (0.52–0.67) 0.49 (0.42–0.65) 0.59 (0.50–0.65)
   30 чел.  LUNA + REVE 0.62 (0.40–0.91) 0.87 (0.46–1.00)  0.75 (0.66–0.81) 0.83 (0.51–1.00) 0.77 (0.63–0.85)
```

## 8. Итоги

### ROC AUC (95% ДИ, бутстреп по людям; ранги пересчитаны на общих записях: NMT eval 184 + Турция 121)

| Модель / ансамбль (средние ранги) | NMT eval | Турция |
|---|---|---|
| **LUNA-large (3 запуска)** | 0,858 (0,80–0,91) | 0,795 (0,71–0,87) |
| REVE | 0,879 (0,83–0,92) | **0,874 (0,80–0,93)** |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| BIOT | 0,798 (0,73–0,86) | 0,821 (0,73–0,89) |
| DeepEpilepsy | 0,900 (0,85–0,94) | 0,682 (0,59–0,77) |
| LUNA + REVE | 0,894 (0,85–0,93) | 0,852 (0,77–0,91) |
| LUNA + REVE + CBraMod | 0,887 (0,84–0,93) | 0,863 (0,79–0,92) |
| LUNA + REVE + DeepEpilepsy | **0,905 (0,86–0,94)** | 0,840 (0,76–0,90) |

Запуски LUNA: NMT eval 0,86 / 0,86 / 0,85 (0,854 ± 0,004), Турция 0,77 / 0,82 / 0,79 (0,792 ± 0,023). Скорость 1,5·10⁻⁴ выбрана по отложенным 15 % (0,841; 5·10⁻⁵ — 0,837, 5·10⁻⁴ авторов — 0,783).

Парный бутстреп разности ROC AUC (95% ДИ):
- **LUNA − REVE: Турция −0,079 (−0,154…−0,007)** — LUNA значимо хуже; NMT eval −0,021 (−0,073…+0,030).
- LUNA − CBraMod: Турция −0,024 (−0,106…+0,056), NMT eval +0,012 — равны.
- LUNA − BIOT: Турция −0,025 (не значимо); NMT eval +0,060 (+0,014…+0,108).
- LUNA − DeepEpilepsy: Турция +0,114 (+0,016…+0,212); NMT eval −0,042 (−0,087…−0,001).
- LUNA + REVE − REVE: Турция −0,022 (−0,063…+0,015), NMT eval +0,015 (−0,009…+0,042) — ансамбль к REVE ничего не добавляет.

### Пороговые метрики (оценка — среднее вероятностей 3 запусков; порог по отложенным 15 % NMT train, на Турцию перенесён тот же)

| Порог | Набор | Accuracy | Сбаланс. accuracy | Precision | Recall | Специфичность |
|---|---|---|---|---|---|---|
| Юден (0,526) | NMT eval (185) | 0,75 (0,69–0,81) | 0,75 (0,68–0,81) | 0,78 (0,68–0,86) | 0,69 (0,59–0,78) | 0,81 (0,73–0,89) |
| то же, перенесён | Турция (121) | 0,64 (0,55–0,72) | 0,56 (0,52–0,61) | 1,00 | 0,12 (0,04–0,21) | 1,00 |
| чувств. 80 % (0,506) | NMT eval | 0,74 (0,66–0,79) | 0,73 (0,66–0,80) | 0,75 (0,65–0,84) | 0,69 (0,59–0,78) | 0,78 (0,69–0,86) |
| то же, перенесён | Турция | 0,64 (0,55–0,72) | 0,56 (0,52–0,61) | 1,00 | 0,12 (0,04–0,21) | 1,00 |

Порог «чувствительность 80 %» на валидации (0,506) почти совпал с порогом Юдена (0,526), поэтому на NMT eval чувствительность ниже 80 % (0,69).

### Калибровка на месте (Турция): порог по k размеченным людям (поровну классов, Юден), проверка на остальных, 500 повторов (среднее, 2,5–97,5 % повторов)

| k | Модель | Сбаланс. accuracy | Recall | Специфичность | Precision | Accuracy |
|---|---|---|---|---|---|---|
| 10 | LUNA | 0,69 (0,56–0,73) | 0,55 | 0,82 | 0,77 | 0,71 |
| 10 | REVE | 0,75 (0,62–0,81) | 0,62 | 0,88 | 0,85 | 0,78 |
| 20 | **LUNA** | **0,69 (0,61–0,74)** | 0,56 (0,25–0,93) | 0,82 (0,34–1,00) | 0,76 | 0,72 |
| 20 | REVE | 0,77 (0,66–0,82) | 0,65 | 0,88 | 0,84 | 0,79 |
| 20 | CBraMod / BIOT / DeepEpilepsy | 0,69 / 0,73 / 0,60 | | | | |
| 20 | LUNA + REVE | 0,74 (0,66–0,80) | 0,62 | 0,87 | 0,83 | 0,77 |
| 30 | LUNA | 0,69 (0,63–0,75) | 0,56 | 0,83 | 0,76 | 0,73 |
| 30 | REVE | 0,77 (0,68–0,83) | 0,67 | 0,87 | 0,82 | 0,79 |

Полная таблица — `results/luna_turkey_local_threshold_calibration.csv`. Расчёт заново для всех моделей на одних и тех же выборках. Способ из `turkey_local_threshold_calibration.csv` не записан и точно не воспроизвёлся: у REVE при 20 людях здесь 0,77, в старом файле 0,76. Поэтому сравнивать лучше внутри новой таблицы.

## 9. Выводы

1. **LUNA-large переносится хуже REVE**: Турция 0,80 (0,71–0,87) против 0,87, разность −0,08 (ДИ −0,15…−0,01) значима. На NMT eval LUNA 0,86 — наравне с REVE и CBraMod. С CBraMod и BIOT на Турции равна, DeepEpilepsy обходит (+0,11).
2. **В ансамбле с REVE пользы нет**: LUNA + REVE 0,85 на Турции против 0,87 у одной REVE (−0,02, не значимо). На NMT eval +0,015 (не значимо). Оценки LUNA и REVE на Турции согласованы (Спирмен 0,69).
3. **Связь с амплитудой записи слабее всех**: Спирмен +0,19 (REVE +0,31, BIOT +0,25, CBraMod +0,12; одна амплитуда даёт AUC 0,76). Это следствие z-оценки каждого канала, как у авторов. Возможно, меньший AUC на Турции — отчасти потому, что LUNA меньше опирается на амплитуду, а она в этом наборе разделяет группы. Проверить это напрямую мы не можем.
4. **Порог с NMT не переносится** (как у REVE, CBraMod, BIOT): на Турции чувствительность 0,12 при специфичности 1,00. При калибровке на 20 людях сбалансированная accuracy 0,69 — ниже REVE (0,77) и BIOT (0,73), на уровне CBraMod.
5. **Модель недообучена**: лучшая эпоха во всех запусках шестая, последняя; отложенный AUC ещё рос (0,82–0,84), потери падали медленно (0,67 → 0,58). REVE, наоборот, переобучалась после первой эпохи. Возможно, больше эпох (у авторов — до 30 с ранней остановкой) или окна 10 с подняли бы результат. В этом блокноте не проверялось.
6. **Отличия от образцов**: окна 5 с при 256 Гц, фильтр 0,1–75 Гц, биполярный монтаж TCP и z-оценка каждого канала — по коду авторов. До 16 окон по 5 с на запись NMT train — те же 80 с сигнала, что 8 окон по 10 с у REVE и BIOT; до 120 окон по 5 с на запись NMT eval и Турции. Окна выбраны заново, так что записи те же, а окна другие.
7. **Утечка для будущего TUEP**: LUNA предобучена на TUEG + Siena (>21 000 ч). Авторы исключили только субъектов своих задач (TUAB/TUAR/TUSL), так что пациенты TUEP, вероятно, есть в предобучении. Для честной проверки на TUEP нужно исключать пересечение (как у CBraMod, REVE, LaBraM).
8. **Ограничения**: 3 запуска; в Турции 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи. Лицензия весов CC BY-ND 4.0: дообученные веса не сохранялись и никуда не выкладываются.

**Затраты:** блокнот 60,9 мин на T4 (чтение данных ≈20 мин, обучение 40,1 мин) — по квоте Kaggle 17,14 → 18,17 ч (**≈1,0 ч GPU**, бюджет 1,5 ч и лимит 25 ч соблюдены). Короткая проверка на процессоре Kaggle — 5 мин, без GPU.

Файлы: `results/luna_finetune_{runs,summary,per_person}.csv` (в `per_person` есть вероятности на отложенных 15 %, набор `nmt_val`), `results/luna_lr_choice.csv`, `results/luna_threshold_metrics.csv`, `results/luna_ensembles_{summary,per_person}.csv`, `results/luna_paired_differences.csv`, `results/luna_turkey_local_threshold_calibration.csv`.
